# Podcast Listening Time Prediction

This notebook is designed to predict the listening time of podcast episodes based on various features such as episode length, guest popularity, number of ads, and more. The workflow includes data preprocessing, feature engineering, model training, and evaluation. The final predictions are saved in a submission file for further analysis.

#### Import Libraries

In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

#### Read train and test data

In [2]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

#### Describe data

In [3]:
print("Train data's size: ", train_data.shape)
print("Test data's size: ", test_data.shape)

Train data's size:  (750000, 12)
Test data's size:  (250000, 11)


In [4]:
numCols = list(train_data.select_dtypes(exclude='object').columns)

# remove "id" from the category columns
numCols.remove("id")
print(f"There are {len(numCols)} numerical features:\n", numCols)

There are 5 numerical features:
 ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Listening_Time_minutes']


In [5]:
catCols = list(train_data.select_dtypes(include='object').columns)

print(f"There are {len(catCols)} categorical features:\n", catCols)

There are 6 categorical features:
 ['Podcast_Name', 'Episode_Title', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']


<a name="data-preprocessing"></a>
## Data Preprocessing and Feature Engineering

<a name="unused-cat"></a>
## Update Categorical Values

In [25]:
# Update train data to contain values from test data
for col in catCols:
    vals = test_data[col].unique()
    vals_train = train_data[col].unique()
    print(f"{col}: Different values {set(vals) - set(vals_train)}")

Podcast_Name: Different values set()
Episode_Title: Different values set()
Genre: Different values set()
Publication_Day: Different values set()
Publication_Time: Different values set()
Episode_Sentiment: Different values set()


<a name="feature-drop"></a>
## Feature Dropper

In [ ]:
class FeatureDropper(BaseEstimator, TransformerMixin):
    """
    A custom transformer that drops specified features from the dataset.
    """
    
    def fit(self, X, y=None):
        """
        Fit method for the transformer. Does nothing as this transformer does not require fitting.
        """
        return self
    
    def transform(self, data):
        """
        Transform method to drop specified features from the dataset.
        """
        return data.drop(["id", "Podcast_Name", "Episode_Title"], axis=1, errors="ignore")

<a name="feature-add"></a>
## Feature Adder

In [ ]:
class FeatureAdder(BaseEstimator, TransformerMixin):
    """
    A custom transformer that adds new features to the dataset.
    Specifically, it extracts the episode number from the 'Episode_Title' column.
    """
    
    def fit(self, X, y=None):
        """
        Fit method for the transformer. Does nothing as this transformer does not require fitting.
        """
        return self
    
    def transform(self, data):
        """
        Transform method to add new features to the dataset.
        """
        # Extract the episode number from the 'Episode_Title' column and convert it to an integer
        data["Episode_No"] = data["Episode_Title"].str.split("Episode", expand=True)[1].astype(int)
        return data


<a name="modeling"></a>
# Modeling

<a name="data-split"></a>
## Train and Test data split

In [ ]:
# Split the data into training and test sets using StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=22)

# Define the target variable
target = "Listening_Time_minutes"

# Perform the split and separate the indices for training and testing
for train_indices, test_indices in split.split(train_data, train_data[target]):
    # Create training and testing datasets based on the split indices
    train = train_data.loc[train_indices]
    test = train_data.loc[test_indices]

# Separate the target variable from the features for training and testing
y_train = train[target]  # Target variable for training
y_test = test[target]    # Target variable for testing
X_train = train.drop(columns=target)  # Features for training
X_test = test.drop(columns=target)    # Features for testing

<a name="random-forest"></a>
## Random Forest

In [ ]:
def train_and_test_model(search_model, X_train, Y_train, X_test):
    """
    Trains the model using GridSearchCV, finds the best model, and makes predictions on the test set.

    Parameters:
    search_model: GridSearchCV object
        The model wrapped in GridSearchCV for hyperparameter tuning.
    X_train: DataFrame
        The training features.
    Y_train: Series
        The target variable for training.
    X_test: DataFrame
        The test features for making predictions.

    Returns:
    best_model: Estimator
        The best model found by GridSearchCV.
    y_pred: ndarray
        Predictions made by the best model on the test set.
    """
    # Fit the model to the training data
    search_model.fit(X_train, Y_train)

    # Print the cross-validation results
    print(search_model.cv_results_)
    
    # Extract the best model from GridSearchCV
    best_model = search_model.best_estimator_
    print(best_model)
    
    # Make predictions on the test set using the best model
    y_pred = best_model.predict(X_test)

    return best_model, y_pred

In [ ]:
# Uncomment to run the random forest model

# Define the preprocessing pipeline
preprocessor = Pipeline(
    steps=[
        # Add new features using the custom FeatureAdder transformer
        ("feature_add", FeatureAdder()),
        # Drop unnecessary features using the custom FeatureDropper transformer
        ("feature_drop", FeatureDropper()),
        # Apply column transformations: impute missing values and encode categorical features
        ("column_transform", 
         ColumnTransformer(
            transformers=[
                # Impute missing values in numerical columns with the median
                ("numerical_imputer", SimpleImputer(strategy="median"), ["Episode_No", "Episode_Length_minutes", "Guest_Popularity_percentage", "Number_of_Ads"]),
                # One-hot encode categorical columns, ignoring unknown categories
                ("onehot_encoder", OneHotEncoder(handle_unknown="ignore"), ["Genre", "Publication_Day", "Publication_Time", "Episode_Sentiment"])
            ])
        )
    ]
)

# Define the full pipeline with preprocessing and the Random Forest model
pipeline = Pipeline([
    ("preprocessor", preprocessor),  # Add the preprocessing pipeline
    ("randomforest", RandomForestRegressor(warm_start=True, n_jobs=-1, random_state=22, verbose=3))  # Random Forest Regressor
])

# Define the parameter grid for hyperparameter tuning
param_grid = [
    {
        "randomforest__n_estimators": [100],  # Number of trees in the forest
        "randomforest__max_depth": [30]      # Maximum depth of the tree
    }
]

# Use GridSearchCV for hyperparameter tuning with cross-validation
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=3,  # 3-fold cross-validation
    scoring="neg_root_mean_squared_error",  # Scoring metric
    return_train_score=True, 
    n_jobs=-1,  # Use all available processors
    verbose=3  # Verbosity level
)

In [36]:
# best_model, y_pred = train_and_test_model(grid_search, X_train, y_train, X_test)

# accuracy = root_mean_squared_error(y_test, y_pred)
# print(accuracy)
# best_model

In [37]:
_, predictions = train_and_test_model(grid_search, train_data.drop(columns=[target]), train_data[target], test_data)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.


building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
b

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 207.5min finished


Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('feature_add', FeatureAdder()),
                                 ('feature_drop', FeatureDropper()),
                                 ('column_transform',
                                  ColumnTransformer(transformers=[('numerical_imputer',
                                                                   SimpleImputer(strategy='median'),
                                                                   ['Episode_No',
                                                                    'Episode_Length_minutes',
                                                                    'Guest_Popularity_percentage',
                                                                    'Number_of_Ads']),
                                                                  ('onehot_encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'),
                                   

NameError: name 'x_test_final' is not defined

In [ ]:
# Create a submission DataFrame with the "id" column from the test data
submission_df = pd.DataFrame(test_data["id"])

# Add the predicted target values to the submission DataFrame
submission_df[target] = predictions

# Save the submission DataFrame to a CSV file
submission_df.to_csv("data/lightgbm.csv", index=False)

## RMSE: 13.26